In [1]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5] 
q = df[:, 6] 

#p[76:] -= 3
#q[76:] -= 3

err_p = df[:, 8]
err_q =  df[:, 7]



p = np.delete(p, [0, 62, 88] )
q = np.delete(q, [0, 62, 88]  )
err_p = np.delete(err_p, [0, 62, 88] )
err_q = np.delete(err_q, [0, 62, 88] )



#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root_split.png")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      14473.8
NDf                       = 
Parametri stimati:
A = -1.0080348131154342 ± 0.2179267858590797
B = 1.9155972596739395 ± 0.15616923546674824
C = 200.7759102272441 ± 0.09546267995490164
Chi2 = 14473.759824928402
         110
Edm                       =   2.5788e-07
NCalls                    =          138
A                         =     -1.00803   +/-   0.217927    
B                         =       1.9156   +/-   0.156169    
C                         =      200.776   +/-   0.0954627   


Info in <TCanvas::Print>: png file fit_root_split.png has been created


## Fit separati a p = 400



In [2]:

p_cut = 400

model_fit_1 = ROOT.TF1('model_1', model, min(p), p_cut, 3)      # parte con p < 400
model_fit_1.SetParameters(par[0], par[1], par[2])
model_fit_1.SetParNames("A", "B", "C")
model_fit_1.SetLineColor(ROOT.kRed + 1)
model_fit_1.SetLineWidth(2)

model_fit_2 = ROOT.TF1('model_2', model, p_cut, max(p), 3)      # parte con p > 400
model_fit_2.SetParameters(par[0], par[1], par[2])
model_fit_2.SetParNames("A", "B", "C")
model_fit_2.SetLineColor(ROOT.kGreen + 2)
model_fit_2.SetLineWidth(2)

gr_split = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr_split.SetTitle("Fit separati; p; q")
gr_split.SetMarkerStyle(20)
gr_split.SetMarkerSize(0.7)
gr_split.SetMarkerColor(ROOT.kAzure + 2)
gr_split.SetLineColor(ROOT.kAzure + 3)
gr_split.SetLineWidth(1)

fit_res_1 = gr_split.Fit(model_fit_1, "SRN")
fit_res_2 = gr_split.Fit(model_fit_2, "SRN")

par_1 = model_fit_1.GetParameters()
par_2 = model_fit_2.GetParameters()
errors_1 = model_fit_1.GetParErrors()
errors_2 = model_fit_2.GetParErrors()
chi2_1, ndf_1 = fit_res_1.Chi2(), fit_res_1.Ndf()
chi2_2, ndf_2 = fit_res_2.Chi2(), fit_res_2.Ndf()


print(f"Fit 1 (p < {p_cut}):")
print(f"A = {par_1[0]} ± {errors_1[0]}")
print(f"B = {par_1[1]} ± {errors_1[1]}")
print(f"C = {par_1[2]} ± {errors_1[2]}")
print(f"Chi2 = {chi2_1}   ndf = {ndf_1}")
print()
print(f"Fit 2 (p > {p_cut}):")
print(f"A = {par_2[0]} ± {errors_2[0]}")
print(f"B = {par_2[1]} ± {errors_2[1]}")
print(f"C = {par_2[2]} ± {errors_2[2]}")
print(f"Chi2 = {chi2_2}   ndf = {ndf_2}")

print()
for i, nome in enumerate(["A", "B", "C"]):
    diff = np.abs(par_1[i] - par_2[i])
    sigma = np.sqrt(errors_1[i]**2 + errors_2[i]**2)
    if diff < 3*sigma:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> compatibili")
    else:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> NON compatibili")

chi2_tot = model_fit.GetChisquare()      
ndf_tot = model_fit.GetNDF()
chi2_split = chi2_1 + chi2_2            
ndf_split = ndf_1 + ndf_2
print()
print(f"Fit unico    : chi2 = {chi2_tot:.1f}  ndf = {ndf_tot}  chi2/ndf = {chi2_tot/ndf_tot:.1f}")
print(f"Due fit somma: chi2 = {chi2_split:.1f}  ndf = {ndf_split}  chi2/ndf = {chi2_split/ndf_split:.1f}")

Fit 1 (p < 400):
A = -15.262294849204446 ± 1.0531397284068873
B = -43.52858147317508 ± 1.614277550453714
C = 213.58846187850563 ± 0.6420717981888823
Chi2 = 1570.0126559201021   ndf = 73

Fit 2 (p > 400):
A = -8.673913251047976 ± 0.5508388831759221
B = -3.792254695035056 ± 0.28882201013707726
C = 204.4377094556519 ± 0.1970219687004865
Chi2 = 2611.9386060981014   ndf = 34

A: differenza = 6.588 (5.5 sigma) -> NON compatibili
B: differenza = 39.736 (24.2 sigma) -> NON compatibili
C: differenza = 9.151 (13.6 sigma) -> NON compatibili

Fit unico    : chi2 = 14473.8  ndf = 110  chi2/ndf = 131.6
Due fit somma: chi2 = 4182.0  ndf = 107  chi2/ndf = 39.1
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1570.01
NDf                       =           73
Edm                       =  1.23533e-07
NCalls                    =          111
A                         =     -15.2623   +/-   1.05314     
B                         =     -43.5286   +/-   1

In [3]:
c_split = ROOT.TCanvas("c_split", "Fit separati", 800, 600)
c_split.SetGrid()

gr_split.Draw("AP")
model_fit_1.Draw("same")
model_fit_2.Draw("same")

leg_split = ROOT.TLegend(0.40, 0.72, 0.89, 0.89)
leg_split.AddEntry(gr_split, "Dati", "pe")
leg_split.AddEntry(model_fit_1, f"Fit p < {p_cut}: A = {par_1[0]:.1f} #pm {errors_1[0]:.1f}, #chi^{{2}}/ndf = {chi2_1:.0f}/{ndf_1}", "l")
leg_split.AddEntry(model_fit_2, f"Fit p > {p_cut}: A = {par_2[0]:.1f} #pm {errors_2[0]:.1f}, #chi^{{2}}/ndf = {chi2_2:.0f}/{ndf_2}", "l")
leg_split.Draw()

c_split.Draw()
c_split.SaveAs("fit_root_split.png")

Info in <TCanvas::Print>: png file fit_root_split.png has been created


In [4]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [5]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff



#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist_split.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Fit status: 0
Parametri stimati:
const = 63.96419755302867 ± 8.027487474804495
#mu = 4.671411911988578 ± 0.2483863793837035
#sigma = 2.0008682150566566 ± 0.13543551143679167
Chi2 = 32.95106886745551
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      32.9511
NDf                       =           10
Edm                       =  6.86552e-07
NCalls                    =          160
const                     =      63.9642   +/-   8.02749     
mu                        =      4.67141   +/-   0.248386    
sigma                     =      2.00087   +/-   0.135436     	 (limited)


Info in <TCanvas::Print>: png file hist_split.png has been created




-1.560947139141609
66
-8.50884657347015
67
-4.415992710736622
68
-7.2773867077429415
70
-14.758053391815066
71
-24.7354673563883
72
-31.54283024534322
73
-52.15641271616138
74
-63.54373863797072
75
-15.478694338513428
76
-9.824296718854493
91
-2.4343207454161866
96
-10.775443930385936
98
-4.305405917176946
102
-8.481044829162347
103
-4.323093667920824
104
-10.868568407049175
106
-12.189698095197285
107
-6.5117717607665995
108
-6.9536297966602705
110
-1.228437271012943
111
-7.332867833165924
112


In [6]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [7]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line_spli.png")

[ 1.24813638e+02  1.51307918e+02  1.17471246e+02  1.16414333e+02
  1.11553803e+02  1.08673567e+02  9.88823325e+01  9.98754904e+01
  8.79555662e+01  8.34044027e+01  9.20247699e+01  7.40481330e+01
  7.59082794e+01  6.77620125e+01  7.88058643e+01  7.67864941e+01
  7.28328582e+01  6.68283590e+01  6.58273810e+01  5.96389550e+01
  6.18450730e+01  5.79047312e+01  6.02317711e+01  5.40373896e+01
  5.08328090e+01  4.68069881e+01  4.37536938e+01  4.69149196e+01
  4.66328855e+01  4.25812539e+01  4.61854570e+01  4.19552729e+01
  4.03632920e+01  3.80684632e+01  3.66430611e+01  3.47331151e+01
  3.44956947e+01  3.24119575e+01  3.31574183e+01  2.78667903e+01
  2.94676544e+01  2.79362793e+01  2.23361119e+01  1.99966339e+01
  2.45458486e+01  2.14827956e+01  2.23023257e+01  2.39680116e+01
  2.15350318e+01  2.14861241e+01  2.33201333e+01  1.90946890e+01
  2.12097943e+01  1.97639688e+01  2.12088937e+01  1.98110867e+01
  1.95937799e+01  1.60281566e+01  1.55984131e+01  1.50151327e+01
  9.79678610e-01  2.22839

Info in <TCanvas::Print>: png file fit_line_spli.png has been created


In [8]:
mask_1 = p < p_cut
p_1 = p[mask_1]
q_1 = q[mask_1]
err_p_1 = err_p[mask_1]
err_q_1 = err_q[mask_1]

q_attesi_1 = np.array([model_fit_1.Eval(x) for x in p_1])
err_qeff_1 = np.sqrt(err_q_1**2 + ((model_derivative(p_1, par_1)) * err_p_1)**2)

residuals_1 = q_1 - q_attesi_1
residuals_norm_1 = residuals_1 / err_qeff_1

gaussian_fit_1 = ROOT.TF1('gaussian_1', 'gaus', min(residuals_norm_1), max(residuals_norm_1), 3)
gaussian_fit_1.SetLineColor(ROOT.kRed + 1)
gaussian_fit_1.SetLineWidth(3)

hist_1 = ROOT.TH1F("hist_1", "Istogramma residui norm. (p < 400); (q - q_{att})/err; conteggi", 15, min(residuals_norm_1), max(residuals_norm_1))
for val in residuals_norm_1:
    hist_1.Fill(val)
hist_1.SetLineColor(ROOT.kAzure + 3)
hist_1.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)

c_hist_1 = ROOT.TCanvas("c_hist_1", "Hist Residui 1", 800, 600)
hist_1.Fit(gaussian_fit_1, "S")
hist_1.Draw()
c_hist_1.Draw()

line_fit_1 = ROOT.TF1('line_1', line, min(p_1), max(p_1), 2)
line_fit_1.SetParameters(0, 0) # Inizializziamo a 0
line_fit_1.SetLineColor(ROOT.kRed + 1)

gr_res_1 = ROOT.TGraphErrors(len(p_1), p_1, residuals_1, np.zeros_like(p_1), err_qeff_1)
gr_res_1.SetTitle("Residui in funzione di p (p < 400); p; q - q_{att}")
gr_res_1.SetMarkerStyle(20)
gr_res_1.SetMarkerColor(ROOT.kAzure + 2)
gr_res_1.SetLineColor(ROOT.kAzure + 3)

c_res_1 = ROOT.TCanvas("c_res_1", "Residui vs p 1", 800, 600)
c_res_1.SetGrid()
gr_res_1.Fit(line_fit_1, "S")
gr_res_1.Draw("AP")
c_res_1.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7.99731
NDf                       =            7
Edm                       =  8.57729e-08
NCalls                    =          304
Constant                  =      61.0734   +/-   7.91481     
Mean                      =      1.32855   +/-   0.154669    
Sigma                     =     0.967157   +/-   0.0708843    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1568.25
NDf                       =           74
Edm                       =  4.65185e-06
NCalls                    =           42
p0                        =  -0.00347691   +/-   0.00261739  
p1                        =      1.27974   +/-   0.965189    


In [9]:
mask_2 = p >= p_cut
p_2 = p[mask_2]
q_2 = q[mask_2]
err_p_2 = err_p[mask_2]
err_q_2 = err_q[mask_2]

q_attesi_2 = np.array([model_fit_2.Eval(x) for x in p_2])
err_qeff_2 = np.sqrt(err_q_2**2 + ((model_derivative(p_2, par_2)) * err_p_2)**2)

residuals_2 = q_2 - q_attesi_2
residuals_norm_2 = residuals_2 / err_qeff_2

gaussian_fit_2 = ROOT.TF1('gaussian_2', 'gaus', min(residuals_norm_2), max(residuals_norm_2), 3)
gaussian_fit_2.SetLineColor(ROOT.kGreen + 2)
gaussian_fit_2.SetLineWidth(3)

hist_2 = ROOT.TH1F("hist_2", "Istogramma residui norm. (p > 400); (q - q_{att})/err; conteggi", 15, min(residuals_norm_2), max(residuals_norm_2))
for val in residuals_norm_2:
    hist_2.Fill(val)
hist_2.SetLineColor(ROOT.kAzure + 3)
hist_2.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)

c_hist_2 = ROOT.TCanvas("c_hist_2", "Hist Residui 2", 800, 600)
hist_2.Fit(gaussian_fit_2, "S")
hist_2.Draw()
c_hist_2.Draw()

line_fit_2 = ROOT.TF1('line_2', line, min(p_2), max(p_2), 2)
line_fit_2.SetParameters(0, 0)
line_fit_2.SetLineColor(ROOT.kGreen + 2)

gr_res_2 = ROOT.TGraphErrors(len(p_2), p_2, residuals_2, np.zeros_like(p_2), err_qeff_2)
gr_res_2.SetTitle("Residui in funzione di p (p > 400); p; q - q_{att}")
gr_res_2.SetMarkerStyle(20)
gr_res_2.SetMarkerColor(ROOT.kAzure + 2)
gr_res_2.SetLineColor(ROOT.kAzure + 3)

c_res_2 = ROOT.TCanvas("c_res_2", "Residui vs p 2", 800, 600)
c_res_2.SetGrid()
gr_res_2.Fit(line_fit_2, "S")
gr_res_2.Draw("AP")
c_res_2.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      10.1775
NDf                       =            9
Edm                       =   8.6739e-09
NCalls                    =           80
Constant                  =      3.17698   +/-   0.86813     
Mean                      =     -2.07245   +/-   2.81931     
Sigma                     =      10.5016   +/-   3.20205      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2602.44
NDf                       =           35
Edm                       =  6.41391e-16
NCalls                    =           38
p0                        =  0.000189979   +/-   6.16279e-05 
p1                        =     -0.13899   +/-   0.0495646   
